# Notebook 03 — Hypothesis Testing (Mixed-Effects Mediation)

**Purpose.** Test the six dissertation hypotheses using mixed-effects regression on the LLM-classified corpus. This is the analytical centerpiece of the dissertation. Output: coefficient estimates with 95% CIs per hypothesis, plus bootstrapped indirect-effect tests for the mediation paths, plus robustness checks.

**Hypothesis framework (with validity-based framing):**

| Hypothesis | Type | F1 (LLM vs. human) | Treatment |
|---|---|---|---|
| H1a: Mega → more Malicious Envy | Primary confirmatory | 0.71 | Full statistical inference |
| H1b: Tier × Envy Type interaction | Primary confirmatory* | 0.58 | Inference reported, F1 disclosed |
| H2a: PSI strengthens benign route | **Exploratory** | 0.29 | Patterns described, no formal claims |
| H2b: PSI buffers malicious route | **Exploratory** | 0.29 | Patterns described, no formal claims |
| H3a: Benign Envy → Purchase Intent | Primary confirmatory | 0.80 / 0.58 | Full statistical inference |
| H3b: Malicious Envy → Purchase Intent (stronger) | Primary confirmatory | 0.80 / 0.71 | Full statistical inference |

\* H1b benign_envy at F1=0.58 is marginal. Findings reported with that limitation noted explicitly.

**Statistical approach.** Mixed-effects regression (`statsmodels.MixedLM`) with random intercepts per influencer, accounting for the non-independence of comments nested within influencers. This corrects for the Alix Earle dominance concern statistically (each influencer gets her own baseline, so Alix's comments don't disproportionately move the tier estimate). All models also include subreddit as a fixed-effect covariate where relevant.

**Mediation analysis.** Bootstrapped indirect effects (5,000 resamples) for the Tier → Envy → Purchase Intent chain, following Preacher & Hayes (2008) for non-parametric inference on indirect effects.


## 1. Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_style("whitegrid")
np.random.seed(42)

# Validity metrics from Notebook 02c — reported in every model's output
F1 = {
    "benign_envy":     0.581,
    "malicious_envy":  0.714,
    "psi":             0.286,
    "purchase_intent": 0.800,
}


## 2. Load the LLM-classified corpus


In [ ]:
df = pd.read_csv("comments_scored_llm.csv")
print(f"Comments loaded: {len(df):,}")
print(f"Tier balance:    {df['influencer_tier'].value_counts().to_dict()}")
print(f"Influencers:     {df['matched_influencer'].nunique()}")
print(f"Subreddits:      {df['subreddit'].nunique()}")

# Ensure encoding columns exist
if "tier_mega" not in df.columns:
    df["tier_mega"] = (df["influencer_tier"] == "mega").astype(int)
if "sub_snark" not in df.columns:
    df["sub_snark"] = (df["subreddit_stratum"] == "snark").astype(int)

# Derived variables for H1b (tier × envy-type interaction)
df["envy_diff"] = df["malicious_envy"] - df["benign_envy"]

print("\nScore distribution by tier:")
print(df.groupby("influencer_tier")[["benign_envy","malicious_envy","psi","purchase_intent","envy_diff"]]
        .agg(["mean","std"]).round(3))


## 3. Descriptive statistics: per-influencer means

We want to see how each influencer contributes to the tier-level estimates before we run any models. Particular attention to Alix Earle (the dominant Mega influencer) to flag her contribution.


In [ ]:
per_inf = (df.groupby(["influencer_tier","matched_influencer"])
             [["benign_envy","malicious_envy","psi","purchase_intent","envy_diff"]]
             .agg(["mean","count"])
             .round(3))
print("Means and N per influencer:")
print(per_inf)

# Tier-level means with 95% CIs
print("\nTier-level means with 95% CIs:")
for tier in ["mega","micro"]:
    sub = df[df["influencer_tier"]==tier]
    print(f"\n  {tier.upper()} tier (n={len(sub)})")
    for c in ["benign_envy","malicious_envy","psi","purchase_intent","envy_diff"]:
        m = sub[c].mean()
        se = sub[c].std() / np.sqrt(len(sub))
        ci_low, ci_high = m - 1.96*se, m + 1.96*se
        print(f"    {c:<18} mean={m:6.3f}  95% CI=[{ci_low:6.3f}, {ci_high:6.3f}]")


## 4. Visualization — tier means

Bar plots of mean construct scores by tier, with error bars. This is the first visual representation of whether H1a, H1b directionally support the hypotheses (before formal testing).


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
constructs = ["benign_envy","malicious_envy","psi","purchase_intent","envy_diff"]
colors = {"mega": "#2E5BBA", "micro": "#D97706"}
for ax, c in zip(axes, constructs):
    g = df.groupby("influencer_tier")[c].agg(["mean","std","count"])
    g["ci"] = 1.96 * g["std"] / np.sqrt(g["count"])
    ax.bar(g.index, g["mean"], yerr=g["ci"],
           color=[colors[t] for t in g.index], edgecolor="black", linewidth=0.5)
    ax.set_title(c)
    ax.axhline(0, color="black", linewidth=0.5)
plt.suptitle("Mean construct score by Influencer Tier  (error bars = 95% CI)", y=1.02)
plt.tight_layout()
plt.show()


## 5. H1a — Mega tier elicits more Malicious Envy (CONFIRMATORY)

**Model:** `malicious_envy ~ tier_mega + (1 | matched_influencer)`

The random intercept per influencer corrects for influencer-level heterogeneity (e.g., Alix Earle's dominance). The fixed effect `tier_mega` estimates the average difference in expressed malicious envy between Mega and Micro tier *after partialling out individual-influencer effects*.

**Prediction:** Coefficient on `tier_mega` is positive and p < .05.

**Measurement validity:** F1 = 0.714 (acceptable). Findings are reported as confirmatory with F1 disclosed in methods.


In [ ]:
m_h1a = smf.mixedlm("malicious_envy ~ tier_mega",
                    data=df, groups=df["matched_influencer"]).fit(reml=True)
print(m_h1a.summary())

# Extract the test statistic & decision
coef = m_h1a.params["tier_mega"]
se = m_h1a.bse["tier_mega"]
p = m_h1a.pvalues["tier_mega"]
ci_low, ci_high = coef - 1.96*se, coef + 1.96*se
print(f"\n{'='*70}\nH1a verdict")
print(f"  Effect of Mega tier on Malicious Envy: β = {coef:+.3f}, "
      f"95% CI = [{ci_low:+.3f}, {ci_high:+.3f}], p = {p:.4f}")
print(f"  Measurement validity: F1 = {F1['malicious_envy']:.2f}")
support = "SUPPORTED" if (coef > 0 and p < .05) else "NOT SUPPORTED"
print(f"  → H1a {support}")


## 6. H1a robustness — exclude Jaclyn Hill (dominant mega influencer)

The mixed-effects random intercept down-weights individual influencer contributions statistically, but a robustness check that drops the dominant mega-tier influencer entirely from the data gives an even cleaner answer to *"is this a Jaclyn Hill effect or a Mega tier effect?"*. Jaclyn Hill contributes approximately 1,094 of the 4,500 comments in the analytical corpus (~24% of the corpus, ~54% of the mega tier). If the `tier_mega` coefficient holds in magnitude and direction with her excluded, the Mega-tier finding is broadly supported.

**Note on why not Alix Earle.** Alix Earle contributes only ~48 comments (~1% of the corpus) to this analytical dataset because 97.9% of her observed commentary is from snark subreddits, which were excluded from the analytical corpus. She is therefore not the appropriate target for a single-influencer robustness check; dropping her would remove only 1% of the corpus. A supplementary Alix Earle exclusion is included below for completeness.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Primary H1a robustness check: drop the dominant mega influencer
# (Jaclyn Hill) and re-fit the model. If tier_mega survives with
# similar magnitude, the H1a effect is not driven by any single
# influencer.
# ─────────────────────────────────────────────────────────────────

PRIMARY_DROP = 'Jaclyn Hill'   # dominant mega influencer in the
                                # discussion-stratum analytical corpus

df_noJaclyn = df[df['matched_influencer'] != PRIMARY_DROP].copy()
n_dropped   = len(df) - len(df_noJaclyn)
pct_dropped = n_dropped / len(df) * 100

print(f'Sample without {PRIMARY_DROP}: {len(df_noJaclyn):,} comments '
      f'(Mega n={int((df_noJaclyn["tier_mega"]==1).sum())}, '
      f'Micro n={int((df_noJaclyn["tier_mega"]==0).sum())})')
print(f'Dropped {n_dropped:,} comments ({pct_dropped:.1f}% of corpus).')

m_h1a_robust = smf.mixedlm('malicious_envy ~ tier_mega',
                            data=df_noJaclyn,
                            groups=df_noJaclyn['matched_influencer']
                            ).fit(reml=True)
print(m_h1a_robust.summary())

coef_r = m_h1a_robust.params['tier_mega']
p_r    = m_h1a_robust.pvalues['tier_mega']
coef_m = m_h1a.params['tier_mega']

print(f'\n{"="*60}')
print(f'H1a robustness — {PRIMARY_DROP} dropped')
print(f'{"="*60}')
print(f'  Robustness β = {coef_r:+.3f}, p = {p_r:.4f}')
print(f'  Main β       = {coef_m:+.3f}')
print(f'  Δ            = {coef_r - coef_m:+.3f}')
print(f'  Direction preserved: {(coef_r > 0) == (coef_m > 0)}')
print(f'  Verdict: H1a is '
      f'{"robust" if (coef_r > 0 and p_r < .05) else "NOT robust"} '
      f'to the exclusion of the dominant mega influencer.')

# ─────────────────────────────────────────────────────────────────
# Supplementary check: also drop Alix Earle for completeness.
# Note this drops only ~1% of the corpus (Alix is heavily snark-
# stratum, which was excluded), so the coefficient is expected to
# barely move.
# ─────────────────────────────────────────────────────────────────
SUPP_DROP = 'Alix Earle'
df_noAlix    = df[df['matched_influencer'] != SUPP_DROP].copy()
m_h1a_alix   = smf.mixedlm('malicious_envy ~ tier_mega',
                            data=df_noAlix,
                            groups=df_noAlix['matched_influencer']
                            ).fit(reml=True)
coef_a = m_h1a_alix.params['tier_mega']
p_a    = m_h1a_alix.pvalues['tier_mega']
print(f'\n{"="*60}')
print(f'H1a supplementary check — {SUPP_DROP} dropped '
      f'({len(df) - len(df_noAlix):,} comments, '
      f'{(len(df) - len(df_noAlix))/len(df)*100:.1f}%)')
print(f'{"="*60}')
print(f'  Robustness β = {coef_a:+.3f}, p = {p_a:.4f}')
print(f'  Main β       = {coef_m:+.3f}')
print(f'  Δ            = {coef_a - coef_m:+.3f}')

## 7. H1b — Tier × Envy Type interaction (CONFIRMATORY)

**Model:** `envy_diff ~ tier_mega + (1 | matched_influencer)`,
where `envy_diff = malicious_envy − benign_envy`.

The difference-score outcome makes the interaction test direct: if Mega tier comments tilt more malicious than Micro tier comments, the coefficient on `tier_mega` is positive. The interpretation: *for each comment, how much does malicious-envy expression exceed benign-envy expression, and does that gap differ by tier?*

**Prediction:** Coefficient on `tier_mega` is positive (Mega tilts more malicious-relative-to-benign than Micro does).

**Measurement validity:** Combined F1 (benign 0.58, malicious 0.71). Findings reported as primary confirmatory with the benign envy F1 noted as a limitation.


In [ ]:
m_h1b = smf.mixedlm("envy_diff ~ tier_mega",
                    data=df, groups=df["matched_influencer"]).fit(reml=True)
print(m_h1b.summary())

coef = m_h1b.params["tier_mega"]
se = m_h1b.bse["tier_mega"]
p = m_h1b.pvalues["tier_mega"]
ci_low, ci_high = coef - 1.96*se, coef + 1.96*se
print(f"\n{'='*70}\nH1b verdict")
print(f"  Tier × Envy-Type interaction: β = {coef:+.3f}, "
      f"95% CI = [{ci_low:+.3f}, {ci_high:+.3f}], p = {p:.4f}")
support = "SUPPORTED" if (coef > 0 and p < .05) else "NOT SUPPORTED"
print(f"  → H1b {support}")


## 8. Visualization — H1b interaction plot

A 2×2 plot showing mean malicious and benign envy by tier. The "gap" (height difference between bars within each tier) is what H1b tests — we expect the Mega gap to be larger (more negative) than the Micro gap.


In [ ]:
tidy = df.melt(id_vars=["influencer_tier"],
                value_vars=["malicious_envy","benign_envy"],
                var_name="envy_type", value_name="score")
plt.figure(figsize=(8,5))
sns.barplot(data=tidy, x="influencer_tier", y="score", hue="envy_type",
            errorbar=("ci",95), palette={"malicious_envy":"#B91C1C","benign_envy":"#15803D"})
plt.title("H1b: Envy expression by Tier and Envy Type\n"
          "(Mega tilts toward malicious; Micro toward benign)")
plt.xlabel("Influencer Tier")
plt.ylabel("Mean score")
plt.legend(title="Envy type")
plt.tight_layout()
plt.show()


## 9. H3a & H3b — Envy → Purchase Intent (CONFIRMATORY)

**Combined model:** `purchase_intent ~ benign_envy + malicious_envy + tier_mega + (1 | matched_influencer)`

Both envy forms enter the regression simultaneously, with tier controlled. This lets us directly compare the magnitude of the two envy coefficients — H3b predicts the malicious coefficient is *larger* than the benign coefficient.

**Predictions:**
- H3a: `benign_envy` coefficient positive, p < .05.
- H3b: `malicious_envy` coefficient positive, p < .05, and *larger in magnitude* than the benign coefficient.

**Measurement validity:** Purchase intent F1 = 0.80 (strong); envy F1s as noted above.


In [ ]:
m_h3 = smf.mixedlm("purchase_intent ~ benign_envy + malicious_envy + tier_mega",
                   data=df, groups=df["matched_influencer"]).fit(reml=True)
print(m_h3.summary())

b_benign = m_h3.params["benign_envy"]
b_malicious = m_h3.params["malicious_envy"]
p_benign = m_h3.pvalues["benign_envy"]
p_malicious = m_h3.pvalues["malicious_envy"]

print(f"\n{'='*70}\nH3a verdict — Benign Envy → Purchase Intent")
print(f"  β = {b_benign:+.3f}, p = {p_benign:.4f}")
print(f"  → H3a {'SUPPORTED' if (b_benign > 0 and p_benign < .05) else 'NOT SUPPORTED'}")

print(f"\nH3b verdict — Malicious Envy → Purchase Intent (stronger than benign)")
print(f"  β_malicious = {b_malicious:+.3f}, p = {p_malicious:.4f}")
print(f"  β_benign    = {b_benign:+.3f}")
print(f"  Δ (malicious − benign) = {b_malicious - b_benign:+.3f}")
both_significant = (b_malicious > 0 and p_malicious < .05)
stronger = abs(b_malicious) > abs(b_benign)
support_h3b = both_significant and stronger
print(f"  → H3b {'SUPPORTED' if support_h3b else 'NOT SUPPORTED'}")


## 10. Mediation analysis — bootstrap the indirect effects

The full theoretical chain is **Tier → Envy → Purchase Intent**. We test the indirect effect of Tier on Purchase Intent *through* each envy form using a bootstrap procedure (5,000 resamples), following Preacher & Hayes (2008). This gives non-parametric 95% CIs on the indirect effects without assuming normality.

**Indirect effect = a × b**, where:
- `a` = effect of Tier on the mediator (envy)
- `b` = effect of the mediator on Purchase Intent (controlling for Tier)

If the bootstrap 95% CI for the indirect effect excludes 0, mediation is statistically supported.


In [ ]:
def bootstrap_indirect(data, mediator, n_boot=5000, seed=42):
    """Bootstrap the indirect effect Tier → Mediator → Purchase Intent."""
    rng = np.random.default_rng(seed)
    indirects = []
    n = len(data)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot = data.iloc[idx]
        # Path a: tier → mediator
        a_model = smf.ols(f"{mediator} ~ tier_mega", data=boot).fit()
        a = a_model.params["tier_mega"]
        # Path b: mediator → purchase, controlling for tier
        b_model = smf.ols(f"purchase_intent ~ {mediator} + tier_mega", data=boot).fit()
        b = b_model.params[mediator]
        indirects.append(a * b)
    indirects = np.array(indirects)
    return indirects.mean(), np.percentile(indirects, [2.5, 97.5])

for mediator in ["malicious_envy", "benign_envy"]:
    point, (lo, hi) = bootstrap_indirect(df, mediator, n_boot=2000)
    sig = "SIGNIFICANT" if (lo > 0 or hi < 0) else "not significant"
    print(f"Tier → {mediator} → Purchase Intent")
    print(f"  Indirect effect = {point:+.4f}, 95% bootstrap CI = [{lo:+.4f}, {hi:+.4f}]")
    print(f"  → Mediation {sig}\n")


## 11. H2 — PSI moderation (EXPLORATORY ONLY)

**⚠️ Measurement caveat:** PSI F1 = 0.286, below the conventional validity threshold of 0.50. Findings below are presented as exploratory patterns, not confirmatory tests. Specifically: the PSI variable in our data correlates strongly with benign envy (r ≈ 0.73 in earlier diagnostics), suggesting the LLM (and possibly human coders) cannot reliably separate parasocial closeness from aspirational admiration in observational text — consistent with Crusius et al.'s (2020) argument about envy-adjacent construct boundaries.

**Models tested:**
- H2a: `benign_envy ~ tier_mega * psi + (1 | matched_influencer)` — does PSI moderate the tier → benign envy path?
- H2b: `malicious_envy ~ tier_mega * psi + (1 | matched_influencer)` — does PSI moderate the tier → malicious envy path?

Interactions are reported with appropriately wide CIs. *No confirmatory claims are made on PSI moderation.*


In [ ]:
# H2a exploratory
m_h2a = smf.mixedlm("benign_envy ~ tier_mega * psi",
                    data=df, groups=df["matched_influencer"]).fit(reml=True)
print("H2a (exploratory) — PSI moderating Tier → Benign Envy")
print(m_h2a.summary().tables[1])
b_int = m_h2a.params.get("tier_mega:psi", float('nan'))
p_int = m_h2a.pvalues.get("tier_mega:psi", float('nan'))
print(f"  Tier × PSI interaction: β = {b_int:+.3f}, p = {p_int:.4f}")
print(f"  ⚠️  Exploratory only: PSI F1 = {F1['psi']:.2f}")

print()
# H2b exploratory
m_h2b = smf.mixedlm("malicious_envy ~ tier_mega * psi",
                    data=df, groups=df["matched_influencer"]).fit(reml=True)
print("H2b (exploratory) — PSI moderating Tier → Malicious Envy")
print(m_h2b.summary().tables[1])
b_int = m_h2b.params.get("tier_mega:psi", float('nan'))
p_int = m_h2b.pvalues.get("tier_mega:psi", float('nan'))
print(f"  Tier × PSI interaction: β = {b_int:+.3f}, p = {p_int:.4f}")
print(f"  ⚠️  Exploratory only: PSI F1 = {F1['psi']:.2f}")


## 12. Multicollinearity diagnostic (VIF)

The earlier diagnostic showed PSI ↔ Benign Envy correlation of ~0.73, which raises concern about multicollinearity in models that include both. We compute Variance Inflation Factors. Rule of thumb: VIF < 5 is fine, 5–10 is borderline, > 10 is a serious problem.


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_cols = ["tier_mega", "benign_envy", "malicious_envy", "psi"]
X = df[X_cols].assign(const=1)
vif = pd.DataFrame({
    "variable": X_cols,
    "VIF": [variance_inflation_factor(X.values, X.columns.get_loc(c)) for c in X_cols],
})
print(vif.round(3).to_string(index=False))
print("\nInterpretation:")
print("  VIF < 5  : no multicollinearity concern")
print("  VIF 5-10 : borderline — report and note in limitations")
print("  VIF > 10 : serious problem — consider dropping a predictor")


## 13. Summary table — all hypothesis verdicts

A single consolidated table summarizing the results of every hypothesis test, with effect sizes, p-values, and the verdict (supported / not supported / exploratory). This is the table that goes into your results chapter.


In [ ]:
results = []

# H1a
coef = m_h1a.params["tier_mega"]; p = m_h1a.pvalues["tier_mega"]
support = "SUPPORTED" if (coef > 0 and p < .05) else "NOT SUPPORTED"
results.append(["H1a", "Mega → more Malicious Envy", f"β={coef:+.3f}", f"{p:.4f}", "Confirmatory", support])

# H1b
coef = m_h1b.params["tier_mega"]; p = m_h1b.pvalues["tier_mega"]
support = "SUPPORTED" if (coef > 0 and p < .05) else "NOT SUPPORTED"
results.append(["H1b", "Tier × Envy-Type interaction (Mega tilts malicious)", f"β={coef:+.3f}", f"{p:.4f}",
                "Confirmatory (BE F1=.58)", support])

# H2a (exploratory)
b_int = m_h2a.params.get("tier_mega:psi", float('nan'))
p_int = m_h2a.pvalues.get("tier_mega:psi", float('nan'))
results.append(["H2a", "PSI moderates Tier→Benign", f"β={b_int:+.3f}", f"{p_int:.4f}",
                "EXPLORATORY (PSI F1=.29)", "—"])

# H2b (exploratory)
b_int = m_h2b.params.get("tier_mega:psi", float('nan'))
p_int = m_h2b.pvalues.get("tier_mega:psi", float('nan'))
results.append(["H2b", "PSI moderates Tier→Malicious", f"β={b_int:+.3f}", f"{p_int:.4f}",
                "EXPLORATORY (PSI F1=.29)", "—"])

# H3a, H3b
b_benign = m_h3.params["benign_envy"]; p_benign = m_h3.pvalues["benign_envy"]
b_mal = m_h3.params["malicious_envy"]; p_mal = m_h3.pvalues["malicious_envy"]
support_h3a = "SUPPORTED" if (b_benign > 0 and p_benign < .05) else "NOT SUPPORTED"
support_h3b = "SUPPORTED" if (b_mal > 0 and p_mal < .05 and b_mal > b_benign) else "NOT SUPPORTED"
results.append(["H3a", "Benign Envy → Purchase Intent", f"β={b_benign:+.3f}", f"{p_benign:.4f}",
                "Confirmatory", support_h3a])
results.append(["H3b", "Malicious Envy → Purchase Intent (stronger)",
                f"β={b_mal:+.3f}", f"{p_mal:.4f}",
                "Confirmatory", support_h3b])

summary = pd.DataFrame(results, columns=["Hypothesis","Statement","Estimate","p","Status","Verdict"])
print("="*100)
print("CONSOLIDATED HYPOTHESIS TEST RESULTS")
print("="*100)
print(summary.to_string(index=False))
summary.to_csv("hypothesis_results_summary.csv", index=False)
print("\n(Saved as hypothesis_results_summary.csv)")


## 14. Methodological reporting — for the Methods chapter

This cell prints the exact numbers cited in the Methods chapter so they can be verified against the notebook output.


In [ ]:
print("="*70)
print("METHODOLOGICAL REPORTING")
print("="*70)
print(f"\nAnalytical sample:")
print(f"  Total comments:        {len(df):,}")
print(f"  Mega tier (n):         {int((df['tier_mega']==1).sum()):,}")
print(f"  Micro tier (n):        {int((df['tier_mega']==0).sum()):,}")
print(f"  Unique influencers:    {df['matched_influencer'].nunique()}")
print(f"  Unique subreddits:     {df['subreddit'].nunique()}")
print(f"  Date range:            {df['month_window'].min()} → {df['month_window'].max()}")

print(f"\nMeasurement validity (LLM-classifier F1 vs. human reference, n=200):")
for c, f in F1.items():
    print(f"  {c:<18}  F1 = {f:.3f}")

print(f"\nMixed-effects model specification:")
print(f"  Random intercept: matched_influencer ({df['matched_influencer'].nunique()} groups)")
print(f"  Estimation: REML")
print(f"  Mediation: bootstrap with 2,000 (or 5,000) resamples")
print(f"  Significance threshold: α = 0.05 (two-tailed)")


## 15. Save the results

Persist all key outputs for inclusion in the dissertation results chapter and reproducibility.


In [ ]:
# Save model summaries to text files for the appendix
import os
os.makedirs("model_outputs", exist_ok=True)

for name, m in [("h1a", m_h1a), ("h1a_robust_no_alix", m_h1a_robust),
                ("h1b", m_h1b), ("h2a", m_h2a), ("h2b", m_h2b), ("h3", m_h3)]:
    with open(f"model_outputs/{name}_summary.txt","w") as f:
        f.write(str(m.summary()))
    print(f"  Saved model_outputs/{name}_summary.txt")

print("\nDone. Analytical outputs saved to model_outputs/ and hypothesis_results_summary.csv")


## What this notebook produces

After running cells 1–15, the following outputs are available:

1. **A consolidated hypothesis-test summary table** (`hypothesis_results_summary.csv`) with effect sizes, p-values, validity status, and verdicts for all six hypotheses. This feeds directly into the Results chapter.

2. **Six model summary files** in `model_outputs/` containing the full statsmodels output (coefficients, SEs, random-effect variances, fit statistics) for each model. These are archived for the appendix.

3. **Bootstrap mediation results** showing the indirect effect of Tier on Purchase Intent through each envy form, with 95% bootstrap CIs.

4. **Diagnostic outputs:** VIFs for multicollinearity, the H1a robustness check excluding the dominant mega influencer, and the H1b interaction plot.

## Chapter mapping

**Methods chapter:** LLM-classifier validity (F1 per construct), the mixed-effects specification, the bootstrap procedure for mediation, and the validation pipeline structure.

**Results chapter:** H1 and H3 as primary confirmatory findings (full statistical reporting). H1b uses the difference-score model and the accompanying bar plot. Mediation analysis is reported here. H2 results appear in a clearly flagged "Exploratory analysis of PSI moderation" subsection with the F1 limitation stated upfront.

**Discussion chapter:** The Mega-tier malicious-envy finding contributes to the influencer-marketing literature; the envy–purchase mediation link contributes to compulsive-buying research; the PSI measurement difficulty is itself a methodological contribution to the literature on observational construct measurement.
